In [ ]:
#Particle Swarm Optimization
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MATCH = 2
GAP = -2

# Load data
def load_data(file):
    df = pd.read_csv(file)
    return df["sequence"].tolist()

def pad_sequences(seqs):
    max_len = max(len(s) for s in seqs)
    return [s.ljust(max_len, '-') for s in seqs]

# Fitness
def fitness(alignment):
    alignment = np.array([list(seq) for seq in alignment])
    score = 0
    for col in alignment.T:
        unique, counts = np.unique(col, return_counts=True)
        score += max(counts) * MATCH
        if '-' in unique:
            score += GAP * counts[list(unique).index('-')]
    return score

# Initialize swarm
def init_swarm(sequences, n_particles=20):
    return [pad_sequences(sequences) for _ in range(n_particles)]

# Update particle
def update_particle(particle):
    new_particle = []
    for seq in particle:
        seq = list(seq)
        if random.random() < 0.3:
            pos = random.randint(0, len(seq)-1)
            seq.insert(pos, '-')
        new_particle.append("".join(seq))
    return pad_sequences(new_particle)

# PSO
def pso(sequences, particles=20, iterations=100):
    swarm = init_swarm(sequences, particles)
    personal_best = swarm.copy()
    global_best = max(swarm, key=fitness)

    history = []

    for _ in range(iterations):
        for i in range(len(swarm)):
            swarm[i] = update_particle(swarm[i])

            if fitness(swarm[i]) > fitness(personal_best[i]):
                personal_best[i] = swarm[i]

        global_best = max(personal_best, key=fitness)
        history.append(fitness(global_best))

    # Plot
    plt.plot(history)
    plt.title("PSO Fitness Progress")
    plt.xlabel("Iteration")
    plt.ylabel("Fitness")
    plt.show()

    return global_best, fitness(global_best)

# Run
sequences = load_data("config.csv")
best, score = pso(sequences)

print("Best Score:", score)